<a href="https://colab.research.google.com/github/mibucko/car_prices/blob/main/notebooks/01_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

We use this notebook for data exploration and planning of future tasks.

Data import

In [1]:
import pandas as pd
url = "https://raw.githubusercontent.com/mibucko/car_prices/main/data/cars.csv"
df = pd.read_csv(url)
print(df.shape)
print(df.columns.tolist())

(56244, 12)
['make', 'model', 'priceUSD', 'year', 'condition', 'mileage(kilometers)', 'fuel_type', 'volume(cm3)', 'color', 'transmission', 'drive_unit', 'segment']


Analyzing missing values

First we look for "missing like" values, and conclude that such inputs do not exist.

In [2]:
missing_like_values = ["NaN", "NULL", "null", "None", "none", "", ".", "-", " "]
for column in df.columns:
    non_null_series = df[column].dropna().astype(str).str.strip()
    found_missing = non_null_series[non_null_series.isin(missing_like_values)]
    if not found_missing.empty:
        print(f"--- False missing values in: {column} ---")
        print(found_missing.value_counts())
        print("\n")

Then we analyze the type of true missing values and conclude that all missing values are standard NaN values (empty cells).

In [3]:
nan_values = ["NaN", "nan", "NULL", "null", "None", "none", "", ".", "-", " "]
for column in df.columns:
  col_as_str = df[column].astype(str).str.strip()
  missing_series = col_as_str[col_as_str.isin(nan_values)]
  if not missing_series.empty:
        print(f"Total missing: {len(missing_series)}")
        print(missing_series.value_counts())

Total missing: 47
volume(cm3)
nan    47
Name: count, dtype: int64
Total missing: 1905
drive_unit
nan    1905
Name: count, dtype: int64
Total missing: 5291
segment
nan    5291
Name: count, dtype: int64


How many rows contain missing values?

Result: In total, 7,021 rows contain missing values; however, none of the missing values are in the target column (car price).

Recommendation:
- drop rows with >1 missing value; test models with and without deletion of rows with 1 missing value.

In [4]:
isna = df.isna().sum(axis=1)[lambda x: x > 0].value_counts().sort_index()
parts = []
for n_missing, count in isna.items():
    parts.append(f"{count} rows {n_missing} missing values")
print(", ".join(parts) + ".")
print(f"In total, {isna.sum()} rows for potential deletion.")

6800 rows 1 missing values, 220 rows 2 missing values, 1 rows 3 missing values.
In total, 7021 rows for potential deletion.


Analyzing column types and extreme values

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56244 entries, 0 to 56243
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   make                 56244 non-null  object 
 1   model                56244 non-null  object 
 2   priceUSD             56244 non-null  int64  
 3   year                 56244 non-null  int64  
 4   condition            56244 non-null  object 
 5   mileage(kilometers)  56244 non-null  float64
 6   fuel_type            56244 non-null  object 
 7   volume(cm3)          56197 non-null  float64
 8   color                56244 non-null  object 
 9   transmission         56244 non-null  object 
 10  drive_unit           54339 non-null  object 
 11  segment              50953 non-null  object 
dtypes: float64(2), int64(2), object(8)
memory usage: 5.1+ MB


Category columns

The cell below shows an example of how each column was analyzed. Here, we present the code for the 'color' column, but the same format was applied to all other columns.

From the analysis below, we conclude:

Columns with nominal categories are 'make', 'model', 'fuel_type', 'color', 'transmission', 'drive_unit' 'segment'.

Column with ordinal categories is 'condition'.

Recommendation: convert the type of these columns to 'category', as this would be beneficial for data processing in Pandas.

In [6]:
df['color'].nunique()
df['color'].value_counts(normalize=True) * 100

,proportion
color,
black,22.020127
silver,17.913022
blue,14.371311
gray,10.324657
white,9.409004
green,6.953631
other,6.039755
red,4.878743
burgundy,3.602162


Analysis of numerical columns (please see the 4 cells below)

Recommendations:
1. Transform 'year' into 'car_age' for simpler model interpretation.
2. Evaluate model performance on a cleaned dataset by removing rows where:

   - price < $150 (46 rows)

   - price > $140,000 (9 rows)

   - year < 1971 (132 rows)

   - zero km or km >= 900000 (608 rows).

3. Column 'volume(cm3)' contains high values that resulted from adding an extra zero (88 rows). Recommendation: divide these values by 10 (for example, 18000 cm3 becomes 1800 cm3).

In [7]:
print(df['priceUSD'].describe())
print(df['priceUSD'].nlargest(3))
print(df['priceUSD'].nsmallest(3))
print("Price <= 150:", (df['priceUSD'] <= 150).sum())
print("Price >= 140000:", (df['priceUSD'] >= 140000).sum())


count     56244.000000
mean       7415.456440
std        8316.959261
min          48.000000
25%        2350.000000
50%        5350.000000
75%        9807.500000
max      235235.000000
Name: priceUSD, dtype: float64
20787    235235
16430    195000
35457    190141
Name: priceUSD, dtype: int64
5952      48
40252     95
5456     100
Name: priceUSD, dtype: int64
Price <= 150: 46
Price >= 140000: 9


In [8]:
print(df['year'].describe())
print(df['year'].nlargest(3))
print(df['year'].nsmallest(3))
print("Older than 1971:", (df['year'] < 1971).sum())

count    56244.000000
mean      2003.454840
std          8.144247
min       1910.000000
25%       1998.000000
50%       2004.000000
75%       2010.000000
max       2019.000000
Name: year, dtype: float64
7398    2019
7448    2019
7910    2019
Name: year, dtype: int64
5952     1910
43024    1933
43020    1936
Name: year, dtype: int64
Older than 1971: 132


In [9]:
print(df['mileage(kilometers)'].describe())
print(df['mileage(kilometers)'].nlargest(3))
print(df['mileage(kilometers)'].nsmallest(3))
print("Mileage > 900.000 km:", (df['mileage(kilometers)'] >= 900000).sum())
print("Zero km, older than 2015. year:", ((df['mileage(kilometers)'] == 0) & (df['year'] < 2015)).sum())

count    5.624400e+04
mean     2.443956e+05
std      3.210307e+05
min      0.000000e+00
25%      1.370000e+05
50%      2.285000e+05
75%      3.100000e+05
max      9.999999e+06
Name: mileage(kilometers), dtype: float64
1394    9999999.0
1854    9999999.0
5783    9999999.0
Name: mileage(kilometers), dtype: float64
184    0.0
809    0.0
863    0.0
Name: mileage(kilometers), dtype: float64
Mileage > 900.000 km: 432
Zero km, older than 2015. year: 176


In [10]:
print(df['volume(cm3)'].describe())
print(df['volume(cm3)'].nlargest(3))
print(df['volume(cm3)'].nsmallest(3))
print("Volume >= 9000:", ((df['volume(cm3)'] >= 9000)).sum())

count    56197.000000
mean      2104.860615
std        959.201633
min        500.000000
25%       1600.000000
50%       1996.000000
75%       2300.000000
max      20000.000000
Name: volume(cm3), dtype: float64
81      20000.0
354     20000.0
1537    20000.0
Name: volume(cm3), dtype: float64
1449    500.0
2037    500.0
5281    500.0
Name: volume(cm3), dtype: float64
Volume >= 9000: 88
